# Best Pipeline Selection and Frontend Validation

This notebook performs the final pipeline selection based on all evaluation results (RQ1, RQ2, RQ3), saves the manifest, and validates the selected pipeline for frontend deployment.

**Goals:**
1. Load all evaluation results from notebooks 05-07
2. Select the best performing pipeline using `select_best_pipeline()` (weighted composite scoring)
3. Save pipeline manifest via `save_best_pipeline_manifest()` to `config/best_pipeline.yaml`
4. Validate the selected pipeline with a test query
5. Verify page reference resolution and confirm frontend readiness

**Outputs:**
- Best pipeline manifest: `config/best_pipeline.yaml` (read by Streamlit app)
- Composite scores: `data/artifacts/evaluation_results/composite_scores.json`

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
from typing import Dict, List, Any, Tuple

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.config import (
    load_golden_standard, get_experiment_configs, ARTIFACTS_DIR,
    load_best_pipeline_config, save_best_pipeline_manifest, BEST_PIPELINE_PATH,
)
from rag.evaluator import select_best_pipeline
from rag.pipeline import build_index, load_pipeline, run_query

print("Imports successful. Project root:", project_root)
print(f"Best pipeline manifest path: {BEST_PIPELINE_PATH}")

In [ ]:
SMOKE_TEST = True

if SMOKE_TEST:
    print("[SMOKE TEST MODE] Using subset for validation")
    test_companies = ["AkerBP_2024"]
else:
    print("[FULL MODE] Using all companies")
    test_companies = None

## Load All Evaluation Results

In [ ]:
# Load evaluation results from previous notebooks
eval_results_dir = ARTIFACTS_DIR / "evaluation_results"
eval_results_dir.mkdir(parents=True, exist_ok=True)

all_results = {}
result_files = {
    "rq1": "rq1_results.json",
    "rq2": "rq2_results.json",
    "rq3": "rq3_results.json",
}

for rq_key, filename in result_files.items():
    filepath = eval_results_dir / filename
    if filepath.exists():
        with open(filepath, "r", encoding="utf-8") as f:
            all_results[rq_key] = json.load(f)
        print(f"Loaded {rq_key} results from {filename}")
    else:
        print(f"Warning: {filename} not found at {filepath}")

if not all_results:
    print("\nNo evaluation results found. Create notebooks 05-07 first.")
else:
    print(f"\nLoaded results for {len(all_results)} research questions")

## Pipeline Selection via `select_best_pipeline()`

Uses the canonical weighted composite scoring from `rag.evaluator.select_best_pipeline()`:

| Metric | Weight |
|--------|--------|
| Retrieval Recall@15 | 0.20 |
| Token F1 | 0.15 |
| BERTScore | 0.10 |
| Citation F1 | 0.15 |
| Faithfulness | 0.15 |
| LLM Judge | 0.15 |
| Cost Efficiency (inverted) | 0.10 |

In [ ]:
# Merge all per-RQ result dicts into a single {config_name: result} mapping.
# If the same config appears under multiple RQs, keep the first (most detailed) one.
merged_results: Dict[str, Any] = {}
for rq_key, rq_results in all_results.items():
    for config_name, config_data in rq_results.items():
        if config_name not in merged_results:
            merged_results[config_name] = config_data

print(f"Collected {len(merged_results)} unique configurations across all RQs")

# Use the canonical selection function from rag.evaluator
best_config_name, best_config_result = select_best_pipeline(merged_results)
best_agg = best_config_result.get("aggregate", {})

print(f"\nBest pipeline selected: {best_config_name}")

In [ ]:
# Display detailed metrics for the best pipeline
print("=" * 60)
print("BEST PIPELINE SELECTED")
print("=" * 60)
print(f"Configuration: {best_config_name}")

# Walk the structured aggregate to print all available metrics
for category in ["retrieval", "answer_quality", "citation", "hallucination", "efficiency"]:
    cat_metrics = best_agg.get(category, {})
    if cat_metrics:
        print(f"\n  [{category.upper()}]")
        for k, v in cat_metrics.items():
            if isinstance(v, float):
                print(f"    {k:35s}: {v:.4f}")
            else:
                print(f"    {k:35s}: {v}")

print("=" * 60)

In [ ]:
# Also compute a quick leaderboard for all configs (for the composite_scores artifact)
from rag.evaluator import select_best_pipeline as _sbp  # already imported; reuse

# Recompute composite per config to build leaderboard
leaderboard: Dict[str, float] = {}
for cfg_name, cfg_result in merged_results.items():
    agg = cfg_result.get("aggregate", {})
    if agg.get("n_evaluated", 0) == 0:
        continue
    metrics = {}
    for cat in ["retrieval", "answer_quality", "citation", "hallucination", "efficiency"]:
        for k, v in agg.get(cat, {}).items():
            metrics[f"{cat}_{k}"] = float(v) if isinstance(v, (int, float)) else 0.0
    judge_raw = metrics.get("answer_quality_avg_llm_judge", 0.0)
    metrics["answer_quality_avg_llm_judge_normalized"] = max(0.0, min(judge_raw / 5.0 if judge_raw > 1.0 else judge_raw, 1.0))
    avg_cost = metrics.get("efficiency_avg_cost_usd", 0.05)
    metrics["efficiency_cost_inverted"] = max(0.0, 1.0 - min(avg_cost * 10, 1.0))
    weights = {
        "retrieval_avg_recall@15": 0.20,
        "answer_quality_avg_token_f1": 0.15,
        "answer_quality_avg_bertscore": 0.10,
        "citation_avg_citation_f1": 0.15,
        "hallucination_avg_faithfulness": 0.15,
        "answer_quality_avg_llm_judge_normalized": 0.15,
        "efficiency_cost_inverted": 0.10,
    }
    composite = sum(weights.get(k, 0) * metrics.get(k, 0) for k in weights)
    leaderboard[cfg_name] = round(composite, 6)

sorted_scores = sorted(leaderboard.items(), key=lambda x: x[1], reverse=True)

print("\n" + "=" * 60)
print("Composite Scores (all configurations)")
print("=" * 60)
for rank, (name, score) in enumerate(sorted_scores, 1):
    marker = " << BEST" if name == best_config_name else ""
    print(f"{rank:2d}. {name:30s} {score:.4f}{marker}")
print("=" * 60)

## Save Best Pipeline Manifest

In [ ]:
# Save manifest using the canonical config helper -> config/best_pipeline.yaml
exp_configs = get_experiment_configs()

if best_config_name in exp_configs:
    best_pipeline_config = exp_configs[best_config_name]
else:
    print(f"Warning: {best_config_name} not in experiment configs, using result config")
    best_pipeline_config = best_config_result.get("config", {})

# Build a compact metrics summary for the manifest
metrics_summary = {}
for cat in ["retrieval", "answer_quality", "citation", "hallucination", "efficiency"]:
    cat_data = best_agg.get(cat, {})
    for k, v in cat_data.items():
        metrics_summary[f"{cat}_{k}"] = round(float(v), 4) if isinstance(v, (int, float)) else v

selection_reason = (
    f"Highest weighted composite score across RQ1/RQ2/RQ3 evaluations "
    f"(n_evaluated={best_agg.get('n_evaluated', '?')})"
)

manifest_path = save_best_pipeline_manifest(
    config_name=best_config_name,
    config=best_pipeline_config,
    selection_reason=selection_reason,
    metrics_summary=metrics_summary,
)
print(f"Best pipeline manifest saved to: {manifest_path}")

In [ ]:
# Save composite scores for reference
composite_path = eval_results_dir / "composite_scores.json"
with open(composite_path, "w", encoding="utf-8") as f:
    json.dump(dict(sorted_scores), f, ensure_ascii=False, indent=2)

print(f"Composite scores saved to: {composite_path}")

## Frontend Validation

In [ ]:
# Validate that load_best_pipeline_config() round-trips correctly
loaded_config = load_best_pipeline_config()
print("Validating load_best_pipeline_config() round-trip...")
print(f"  _best_pipeline_name : {loaded_config.get('_best_pipeline_name')}")
print(f"  _selection_reason   : {loaded_config.get('_selection_reason')[:80]}...")
print(f"  chunking_strategy   : {loaded_config.get('chunking_strategy')}")
print(f"  retrieval_method    : {loaded_config.get('retrieval_method')}")
print(f"  use_reranker        : {loaded_config.get('use_reranker')}")
print(f"  top_k               : {loaded_config.get('top_k')}")

assert loaded_config.get("_best_pipeline_name") == best_config_name, \
    f"Round-trip mismatch: {loaded_config.get('_best_pipeline_name')} != {best_config_name}"
print("\nRound-trip validation PASSED")

In [ ]:
# Build the pipeline index using the loaded (round-tripped) config
pipeline_config = {k: v for k, v in loaded_config.items() if not k.startswith("_")}

print("Building index for best pipeline...")
vector_index, bm25_index, chunks = build_index(
    config=pipeline_config,
    force_rebuild=False,
    companies=test_companies,
)
print(f"Index built: {len(chunks)} chunks")
print(f"Vector index: {vector_index}")
print(f"BM25 index : {bm25_index}")

In [ ]:
# Run a test query through the pipeline
test_query = "What are the company's emissions reduction targets and interim milestones?"
test_company = test_companies[0] if test_companies else "Shell_2024"

print(f"\nRunning test query:")
print(f"  Query: {test_query}")
print(f"  Company: {test_company}")
print("\nGenerating answer...")

try:
    answer_result = run_query(
        query=test_query,
        companies=[test_company],
        esrs_standard=None,
        retriever=None,
        config=pipeline_config
    )
    print(f"\n" + "="*60)
    print("GENERATED ANSWER")
    print("="*60)
    print(answer_result.get("answer", "[No answer generated]"))
    print(f"\nPage References:")
    page_refs = answer_result.get("page_references", [])
    for ref in page_refs:
        print(f"  - Pages {ref.get('pages')}: {ref.get('source', 'Unknown')}")
except Exception as e:
    print(f"Error generating answer: {e}")
    import traceback
    traceback.print_exc()

## Verify Page Reference Resolution

In [ ]:
# Verify that page references come from actual chunks
print("\nVerifying page reference resolution...")
if answer_result:
    page_refs = answer_result.get("page_references", [])
    
    if page_refs:
        chunk_pages = set()
        for chunk in chunks:
            # Extract page numbers from chunk metadata
            if isinstance(chunk, dict):
                pages = chunk.get("page_span", []) or chunk.get("page_number")
            else:
                pages = getattr(chunk, "page_span", []) or getattr(chunk, "page_number", None)
            
            if isinstance(pages, (list, tuple)):
                chunk_pages.update(pages)
            elif isinstance(pages, int):
                chunk_pages.add(pages)
        
        print(f"\nChunks in index have pages: {sorted(chunk_pages)[:20]}..." if len(chunk_pages) > 20 else f"Chunks in index have pages: {sorted(chunk_pages)}")
        
        cited_pages = set()
        for ref in page_refs:
            pages = ref.get("pages", [])
            if isinstance(pages, list):
                cited_pages.update(pages)
        
        print(f"\nCited pages in answer: {sorted(cited_pages)}")
        
        # Check if all cited pages exist in chunks
        invalid_pages = cited_pages - chunk_pages
        if invalid_pages:
            print(f"\nWarning: {len(invalid_pages)} cited pages not found in chunks: {invalid_pages}")
        else:
            print(f"\nSuccess: All {len(cited_pages)} cited pages exist in chunks")
    else:
        print("No page references in answer")
else:
    print("No answer result to verify")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("PIPELINE SELECTION SUMMARY")
print("=" * 60)
print(f"\nSelected Pipeline : {best_config_name}")
print(f"Manifest Location : {manifest_path}")
print(f"Manifest Readable : {BEST_PIPELINE_PATH.exists()}")
print(f"\nLeaderboard (top 3):")
for i, (name, score) in enumerate(sorted_scores[:3], 1):
    print(f"  {i}. {name:30s} {score:.4f}")

# Print key aggregate metrics
for cat_label, cat_key in [
    ("Retrieval", "retrieval"),
    ("Answer Quality", "answer_quality"),
    ("Citations", "citation"),
    ("Hallucination", "hallucination"),
    ("Efficiency", "efficiency"),
]:
    cat_data = best_agg.get(cat_key, {})
    if cat_data:
        items = ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                         for k, v in cat_data.items())
        print(f"\n  [{cat_label}] {items}")

print(f"\nFrontend Ready: Yes")
print("=" * 60)